# Fetch API

The **Fetch API** is a built-in, promise-based interface for making asynchronous HTTP requests in browsers and modern Node.js environments (native since Node 18). It is the modern replacement for `XMLHttpRequest`.

---

## 1. Basic GET Request (async/await)

Reading data requires a **two-step mechanism**, because `fetch()` resolves with a stream-based `Response` object rather than the data itself. You must explicitly resolve the body into a readable format.

```javascript
async function fetchUserData() {
  try {
    // 1. Send the request
    const response = await fetch('https://jsonplaceholder.typicode.com/users/1');

    // 2. Always check the HTTP status code (200–299)
    if (!response.ok) {
      throw new Error(`HTTP error! Status: ${response.status}`);
    }

    // 3. Parse the stream into JSON
    const data = await response.json();
    console.log(data);

  } catch (error) {
    // Catches network errors and manually thrown errors
    console.error('Fetch failed:', error);
  }
}

fetchUserData();
```

**Why two awaits?** The first resolves when response *headers* arrive — status code, content type, and so on are available, but the body may still be downloading. The second waits for the body to finish streaming and parses it. This split is what makes it possible to check `response.status` and bail before downloading a large payload you don't want.

---

## 2. POST Request (Sending Data)

Pass a configuration object as the second argument specifying the method, headers, and a stringified payload.

```javascript
async function createNewUser() {
  const url = 'https://example.com/api/users';
  const userData = { name: 'Jane Doe', role: 'Developer' };

  try {
    const response = await fetch(url, {
      method: 'POST',
      headers: {
        'Content-Type': 'application/json',      // Critical for JSON payloads
        'Authorization': 'Bearer YOUR_TOKEN_HERE'
      },
      body: JSON.stringify(userData)             // Payload as a JSON string
    });

    if (!response.ok) {
      throw new Error(`Failed to create resource: ${response.status}`);
    }

    const result = await response.json();
    console.log('Success:', result);

  } catch (error) {
    console.error('Error during POST:', error);
  }
}
```

Forgetting `JSON.stringify()` is the most common mistake here — passing a raw object gives the server the literal string `[object Object]`.

---

## 3. Alternative Promise Syntax (`.then()`)

The same request without `async`/`await`:

```javascript
fetch('https://example.com/api/data')
  .then(response => {
    if (!response.ok) {
      throw new Error('Network response was not ok');
    }
    return response.json();   // Returns a secondary promise
  })
  .then(data => console.log(data))
  .catch(error => console.error('Problem with fetch operation:', error));
```

Returning `response.json()` from the first `.then()` is what makes the second one receive parsed data rather than a promise.

---

## ⚠️ Crucial Gotcha: Error Handling

**A `fetch()` promise does not reject on HTTP errors.** A `404` or `500` still resolves successfully. The promise only rejects on a complete network breakdown, DNS failure, or CORS block.

```javascript
// WRONG — a 500 sails straight through
const data = await fetch(url).then(r => r.json());

// RIGHT
const response = await fetch(url);
if (!response.ok) throw new Error(`HTTP ${response.status}`);
const data = await response.json();
```

This is deliberate: `fetch` reports on *transport*, and an HTTP error is a successfully delivered message that happens to say "no." Axios takes the opposite stance and rejects on 4xx/5xx, which is a large part of why people still reach for it.

---

## The Response Object

| Property / method | Purpose |
| --- | --- |
| `response.ok` | `true` for any 2xx status. |
| `response.status` | Numeric code, e.g. `404`. |
| `response.statusText` | Text reason, e.g. `"Not Found"`. |
| `response.headers` | A `Headers` object — read with `.get('content-type')`. |
| `response.json()` | Parses the body as JSON. |
| `response.text()` | Returns the body as a plain string. |
| `response.blob()` | Binary data — images, files, PDFs. |
| `response.formData()` | Parses multipart form responses. |

**The body can only be read once.** Each of these consumes the stream, so calling `.json()` after `.text()` throws. If you need it twice, use `response.clone()` first.

```javascript
// Useful when a failing endpoint may return HTML instead of JSON
const text = await response.text();
try {
  const data = JSON.parse(text);
} catch {
  console.error('Server returned non-JSON:', text.slice(0, 200));
}
```

---

## Common Options

```javascript
await fetch(url, {
  method: 'PATCH',
  headers: { 'Content-Type': 'application/json' },
  body: JSON.stringify(payload),
  credentials: 'include',   // send cookies cross-origin
  mode: 'cors',             // 'cors' | 'no-cors' | 'same-origin'
  cache: 'no-store',
  signal: controller.signal // for cancellation — see below
});
```

### Sending form data

Do **not** set `Content-Type` manually with `FormData` — the browser must add the multipart boundary itself.

```javascript
const form = new FormData();
form.append('file', fileInput.files[0]);
form.append('patientId', '42');

await fetch('/api/upload', { method: 'POST', body: form });
```

---

## Cancelling a Request

Use `AbortController`. This matters in React, where a component can unmount before its request returns.

```javascript
const controller = new AbortController();

// Abort automatically after 5 seconds
const timeout = setTimeout(() => controller.abort(), 5000);

try {
  const response = await fetch(url, { signal: controller.signal });
  clearTimeout(timeout);
  const data = await response.json();
} catch (error) {
  if (error.name === 'AbortError') {
    console.log('Request cancelled or timed out');
  } else {
    throw error;
  }
}
```

`fetch` has **no built-in timeout** — without this, a hanging request waits indefinitely.

---

## Parallel Requests

Independent requests should run concurrently, not sequentially.

```javascript
// Sequential — total time is additive
const patients = await fetch('/api/patients').then(r => r.json());
const doctors  = await fetch('/api/doctors').then(r => r.json());

// Concurrent — bound by the slowest single request
const [patients, doctors] = await Promise.all([
  fetch('/api/patients').then(r => r.json()),
  fetch('/api/doctors').then(r => r.json())
]);
```

Use `Promise.allSettled()` instead when partial failure is acceptable — `Promise.all` rejects the whole batch if any single request fails.

---

## A Reusable Wrapper

Once you've written the same `response.ok` check five times, extract it:

```javascript
async function apiRequest(endpoint, options = {}) {
  const response = await fetch(`${BASE_URL}${endpoint}`, {
    headers: {
      'Content-Type': 'application/json',
      ...(options.body && { 'Content-Type': 'application/json' }),
      ...options.headers
    },
    ...options
  });

  if (!response.ok) {
    const message = await response.text();
    throw new Error(`HTTP ${response.status}: ${message}`);
  }

  return response.status === 204 ? null : response.json();
}

// Usage
const patients = await apiRequest('/patients');
const created  = await apiRequest('/patients', {
  method: 'POST',
  body: JSON.stringify({ name: 'Test' })
});
```

The `204` check matters — calling `.json()` on an empty body throws, and `DELETE` endpoints commonly return `204 No Content`.

---

## Reference Links

- [Using Fetch — MDN](https://developer.mozilla.org/en-US/docs/Web/API/Fetch_API/Using_Fetch)
- [Response — MDN](https://developer.mozilla.org/en-US/docs/Web/API/Response)
- [AbortController — MDN](https://developer.mozilla.org/en-US/docs/Web/API/AbortController)
- [Fetch in Node.js](https://nodejs.org/learn/getting-started/fetch)